# Aya (Person 3): Evaluation, Moroccan flood, Under the Clouds

**Goal:** honest accuracy (IoU, F1) of the CNN vs the threshold baseline on the real Moroccan flood (Loukkos / Ksar El Kebir, 21 Jan to 8 Feb 2026), using the Zenodo "SAR Morocco Flood Dataset" as the reference.

**How to use:** open in Google Colab and run the cells top to bottom (Runtime > Run all also works).
**This notebook never pushes to GitHub.** At the end it gives you a zip of the `experiments/` folder; you add those files to the repo yourself from VS Code.

**Honest warnings (they go in the report):**
1. The Zenodo masks were made with HSBA + Otsu, so they are semi-automatic and can favour the Otsu baseline.
2. They are water masks from patch dates, not a flood-change map. We compare *water after*.

**Notes for whoever edits this notebook (VS Code / Copilot):**
- Cells are meant to run in order in a fresh Colab runtime. Do not add `git push`, tokens or passwords.
- Keep these variable names, later cells use them: `BEFORE_TIF`, `AFTER_TIF`, `CKPT`, `DATA_DIR`, `mask_files`, `image_files`, `ref`, `results`, `cnn_water`, `base_water`, `vv_after`.
- Outputs go to `experiments/` only. Never write the 3 GB data or `*.pt` files there.
- The CNN is scored only if `experiments/predict_water.py` exists (Copilot task 1 in the team prompt).

## STEP 0: Config (edit only this cell)

In [ ]:
REPO_URL    = "https://github.com/rabaaab/under-the-clouds.git"   # check this is your team repo URL (must be public, or the clone will fail)
REPO_DIR    = "under-the-clouds"
ZENODO_ID   = "22964153"

BEFORE_DATE = "2026-01-21"            # Ikram's scenes
AFTER_DATE  = "2026-02-08"
WINDOW_DAYS = 6                       # keep reference patches within +-6 days of AFTER_DATE (if a date can be found)
USE_DATE_FILTER = True                # if no patch is inside the window, the notebook retries without the filter and says so

MASK_REGEX  = r"mask|label"           # how mask files are recognised (checked in STEP 4)

## STEP 1: Get the code and install packages
Read only: nothing here changes the repo.

In [ ]:
import os, sys, subprocess, re, json, glob, shutil, random, datetime as dt
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def sh(cmd, show=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if show and out:
        print(out[-2500:])
    return r.returncode

def _is_repo(p):
    return (Path(p) / "src" / "utc").is_dir()

# Work inside the repo: already there, one level up (notebook in notebooks/), or clone it.
if _is_repo("."):
    pass
elif _is_repo(".."):
    os.chdir("..")
elif _is_repo(REPO_DIR):
    os.chdir(REPO_DIR)
else:
    rc = sh(f"git clone {REPO_URL} {REPO_DIR}")
    if rc == 0 and Path(REPO_DIR).is_dir():
        os.chdir(REPO_DIR)
    else:
        print("!! Could not clone the repo (wrong URL, or the repo is private).")
        print("   The notebook continues in", Path.cwd(), "and will take before/after/unet from the shared Drive folder.")
        print("   The CNN step needs the repo code: fix REPO_URL and re-run this cell.")
if Path(".git").is_dir():
    sh("git pull --ff-only")

sh("pip install -q rasterio scikit-image reportlab pandas matplotlib requests google-api-python-client")
if Path("requirements.txt").exists():
    sh("pip install -q -r requirements.txt")

# Keep big files out of git (local only, nothing to commit)
if Path(".git").is_dir():
    ex = Path(".git/info/exclude")
    old = ex.read_text() if ex.exists() else ""
    add = [l for l in ["data/", "*.zip", "*.pt", "outputs/aya_tmp/"] if l not in old]
    if add:
        ex.write_text(old.rstrip("\n") + "\n" + "\n".join(add) + "\n")
Path("experiments/failures").mkdir(parents=True, exist_ok=True)
Path("outputs/aya_tmp").mkdir(parents=True, exist_ok=True)
print("Working folder:", Path.cwd())

## STEP 2: Find the team files (before.tif, after.tif, unet.pt)
Looks in the repo first, then in your **Google Drive**. If a file is only "Shared with me", right-click it in Drive and choose **Make a copy** so it is in My Drive.

In [ ]:
def find_files(name_regex, skip=("data/", ".git/")):
    hits = []
    for p in Path(".").rglob("*"):
        s = str(p).replace("\\", "/")
        if not p.is_file() or any(k in s for k in skip):
            continue
        if re.search(name_regex, p.name, re.I):
            hits.append(p)
    return sorted(hits)

cand_before = find_files(r"^before.*\.tif{1,2}$")
cand_after  = find_files(r"^after.*\.tif{1,2}$")
cand_ckpt   = find_files(r"\.pt$|\.pth$")

# ---- Team shared Drive folder (used only for what is missing in the repo) ----
DRIVE_FOLDER_ID = "14d5-8yBSUm2NuTfJX3XxVB7kwtVqBMxr"   # the shared folder
PREFER_PATH     = "event"                                # prefer files inside a folder whose name contains this

if IN_COLAB and not (cand_before and cand_after and cand_ckpt):
    from google.colab import auth
    auth.authenticate_user()                              # log in with YOUR Google account (it has access to the shared folder)
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    svc = build("drive", "v3")

    def list_children(fid):
        items, token = [], None
        while True:
            r = svc.files().list(q=f"'{fid}' in parents and trashed=false",
                                 fields="nextPageToken, files(id,name,mimeType,size)", pageSize=1000, pageToken=token,
                                 supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            items += r["files"]; token = r.get("nextPageToken")
            if not token: break
        return items

    def walk(fid, path=""):
        for it in list_children(fid):
            if it["mimeType"] == "application/vnd.google-apps.folder":
                yield from walk(it["id"], path + it["name"] + "/")
            else:
                yield path + it["name"], it

    drive_files = list(walk(DRIVE_FOLDER_ID))
    print("Files in the shared Drive folder:")
    for path, it in drive_files:
        print("  ", path, round(int(it.get("size", 0)) / 1e6, 1), "MB")

    TMP = Path("outputs/aya_tmp"); TMP.mkdir(parents=True, exist_ok=True)
    def pick_and_download(pattern, dest_name):
        hits = [(p, it) for p, it in drive_files if re.search(pattern, p.split("/")[-1], re.I)]
        if not hits:
            print("NOT in the Drive folder:", pattern); return None
        hits.sort(key=lambda h: PREFER_PATH.lower() not in h[0].lower())   # preferred folder first
        path, it = hits[0]
        if len(hits) > 1:
            print(f"  {len(hits)} matches for {pattern}, using: {path}")
        out = TMP / dest_name
        req = svc.files().get_media(fileId=it["id"], supportsAllDrives=True)
        with open(out, "wb") as fh:
            dl = MediaIoBaseDownload(fh, req, chunksize=50 * 1024 * 1024)
            done = False
            while not done:
                _, done = dl.next_chunk()
        print("downloaded:", path, "->", out)
        return out

    if not cand_ckpt:
        p = pick_and_download(r"unet.*\.(pt|pth)$", "unet.pt");        cand_ckpt = [p] if p else []
    if not cand_before:
        p = pick_and_download(r"^before.*\.tif{1,2}$", "before.tif");  cand_before = [p] if p else []
    if not cand_after:
        p = pick_and_download(r"^after.*\.tif{1,2}$", "after.tif");    cand_after = [p] if p else []

print("before:", cand_before, "| after:", cand_after, "| ckpt:", cand_ckpt)
BEFORE_TIF = cand_before[0] if cand_before else None
AFTER_TIF  = cand_after[0]  if cand_after  else None
CKPT       = cand_ckpt[0]   if cand_ckpt   else None
print("USING:", BEFORE_TIF, AFTER_TIF, CKPT)
assert BEFORE_TIF and AFTER_TIF, "before.tif / after.tif not found in the repo or in the shared Drive folder (look at the file list printed above)."
if CKPT is None:
    print("WARNING: no checkpoint found. Baseline numbers will work, the CNN needs unet.pt.")

## STEP 3: Download the Zenodo dataset (3 GB, takes a few minutes)

In [ ]:
import requests, zipfile, tarfile, time
RAW = Path("data/zenodo_raw");     RAW.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/SAR_Morocco"); DATA_DIR.mkdir(parents=True, exist_ok=True)

def list_tifs(root):
    """All .tif/.tiff under root, ignoring macOS junk (__MACOSX, ._files) that rasterio cannot open."""
    found = list(Path(root).rglob("*.tif")) + list(Path(root).rglob("*.tiff"))
    return sorted(p for p in found if "__MACOSX" not in p.parts and not p.name.startswith("._"))

rec = None
for attempt in range(4):
    try:
        resp = requests.get(f"https://zenodo.org/api/records/{ZENODO_ID}", timeout=60)
        resp.raise_for_status()
        rec = resp.json()
        break
    except Exception as e:
        print(f"Zenodo record request failed (try {attempt + 1}/4): {e}")
        time.sleep(5 * (attempt + 1))
if rec is None:
    raise RuntimeError("Could not reach Zenodo. Check the internet connection in Colab and run this cell again.")

files = rec.get("files", [])
print("Files in the Zenodo record:")
for f in files:
    print("  ", f.get("key") or f.get("filename"), round((f.get("size") or f.get("filesize") or 0) / 1e6, 1), "MB")
if not files:
    print("!! The record lists no files. Open https://zenodo.org/records/" + ZENODO_ID + " and check.")

for f in files:
    key = f.get("key") or f.get("filename")
    size = f.get("size") or f.get("filesize") or 0
    links = f.get("links") or {}
    url = links.get("self") or links.get("download") or f"https://zenodo.org/records/{ZENODO_ID}/files/{key}?download=1"
    dest = RAW / key
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and (not size or dest.stat().st_size == size):
        print("already downloaded:", key); continue
    for attempt in range(3):
        try:
            print("downloading", key, "...")
            part = dest.with_name(dest.name + ".part")
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                with open(part, "wb") as fh:
                    for chunk in r.iter_content(1 << 20):
                        fh.write(chunk)
            part.replace(dest)
            break
        except Exception as e:
            print(f"  download failed (try {attempt + 1}/3): {e}")
            time.sleep(10)
    else:
        raise RuntimeError(f"Could not download {key}. Run this cell again (finished files are skipped).")

if not list_tifs(DATA_DIR):
    for z in sorted(RAW.rglob("*")):
        if not z.is_file():
            continue
        name = z.name.lower()
        if name.endswith(".zip"):
            print("unzipping", z.name)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(DATA_DIR)
        elif name.endswith((".tar", ".tar.gz", ".tgz", ".tar.bz2", ".tar.xz")):
            print("untarring", z.name)
            with tarfile.open(z) as tf:
                tf.extractall(DATA_DIR)
        elif z.suffix.lower() in (".tif", ".tiff"):
            shutil.copy(z, DATA_DIR / z.name)
    for p in RAW.rglob("*"):
        if p.is_file() and p.suffix.lower() in (".csv", ".md", ".txt", ".json"):
            shutil.copy(p, DATA_DIR / p.name)
n_tif = len(list_tifs(DATA_DIR))
print("tif files:", n_tif)
if n_tif == 0:
    print("!! No .tif found after unpacking. Files in", RAW, ":", [p.name for p in RAW.rglob("*") if p.is_file()][:20])

## Helpers (pure functions, used by the next steps)

In [ ]:
import numpy as np

def to_db(a):
    """Return dB values. If the array looks linear (all >= 0), convert with 10*log10."""
    a = a.astype("float32")
    finite = a[np.isfinite(a)]
    if finite.size and finite.min() >= 0:
        a = 10.0 * np.log10(np.maximum(a, 1e-6))
    return a

def parse_date(text):
    m = re.search(r"(20\d{2})[-_]?(\d{2})[-_]?(\d{2})", str(text))
    if not m:
        return None
    try:
        return dt.date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
    except ValueError:
        return None

def norm_key(name):
    s = Path(str(name)).stem.lower()
    s = re.sub(r"[_\-]?(mask|label|image|img|sar|s1)[_\-]?", "_", s)
    return s.strip("_-")

def scores(pred, ref, valid):
    """IoU, F1, precision, recall for binary maps, only where valid is True."""
    p = (pred > 0) & valid
    r = (ref > 0) & valid
    tp = int((p & r).sum()); fp = int((p & ~r & valid).sum()); fn = int((~p & r & valid).sum())
    d_iou = tp + fp + fn
    d_f1 = 2 * tp + fp + fn
    return {
        "iou": tp / d_iou if d_iou else float("nan"),
        "f1": 2 * tp / d_f1 if d_f1 else float("nan"),
        "precision": tp / (tp + fp) if (tp + fp) else float("nan"),
        "recall": tp / (tp + fn) if (tp + fn) else float("nan"),
        "tp": tp, "fp": fp, "fn": fn, "pixels_scored": int(valid.sum()),
    }

def tile_errors(pred, ref, valid, size=256, step=128):
    rows = []
    H, W = pred.shape
    for y in range(0, max(H - size, 0) + 1, step):
        for x in range(0, max(W - size, 0) + 1, step):
            v = valid[y:y+size, x:x+size]
            if v.size == 0 or v.mean() < 0.5:
                continue
            p = pred[y:y+size, x:x+size] > 0
            r = ref[y:y+size, x:x+size] == 1
            rows.append((y, x, int((p & ~r & v).sum()), int((~p & r & v).sum())))
    return rows

def pick_tiles(rows, k_fp=3, k_fn=2, size=256):
    chosen = []
    def far(y, x):
        return all(abs(y - cy) >= size or abs(x - cx) >= size for _, cy, cx in chosen)
    for kind, idx, k in (("false_alarm", 2, k_fp), ("missed_water", 3, k_fn)):
        n = 0
        for row in sorted(rows, key=lambda r: -r[idx]):
            if row[idx] <= 0 or n >= k:
                break
            if far(row[0], row[1]):
                chosen.append((kind, row[0], row[1])); n += 1
    return chosen

def fmt(v):
    return "n/a" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}"

def honest_text(res):
    """Draft paragraph from real numbers only. res = {'baseline': {...}, 'cnn': {...} or None}"""
    b, c = res.get("baseline"), res.get("cnn")
    parts = []
    if b and c:
        diff = c["iou"] - b["iou"]
        if diff > 0.02:
            verdict = "the CNN scored higher than the threshold baseline"
        elif diff < -0.02:
            verdict = "the threshold baseline scored higher than the CNN"
        else:
            verdict = "the CNN and the threshold baseline scored about the same"
        parts.append(f"On the Moroccan reference, {verdict} (IoU {fmt(c['iou'])} vs {fmt(b['iou'])}, "
                     f"F1 {fmt(c['f1'])} vs {fmt(b['f1'])}).")
    elif b:
        parts.append(f"Only the threshold baseline could be scored so far (IoU {fmt(b['iou'])}, F1 {fmt(b['f1'])}); the CNN is not scored yet.")
    parts.append("These numbers are limited: the reference masks come from Otsu/HSBA thresholding plus visual checks, "
                 "so they are not expert ground truth and may favour the threshold baseline; they are water masks from "
                 "nearby dates, not a flood-change map; and the patches cover only part of the study area.")
    parts.append("Known failure types to describe from the saved crops: false water on the sea and coast, thin lines at tile seams, "
                 "and missed flooded vegetation (grey, not black, in radar).")
    return " ".join(parts)

def clean_json(o):
    """Replace NaN/inf by None so the JSON file is valid everywhere."""
    if isinstance(o, dict):
        return {k: clean_json(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean_json(v) for v in o]
    if isinstance(o, float) and not np.isfinite(o):
        return None
    return o

## STEP 4: Inspect the dataset (dates, units, masks) and write your event notes
Read the printout carefully. It answers: **which dates, what CRS, are values in dB, which files are masks.**

In [ ]:
tifs = list_tifs(DATA_DIR)
csvs = sorted(DATA_DIR.rglob("*.csv")); mds = sorted(DATA_DIR.rglob("*.md"))
mask_files  = [p for p in tifs if re.search(MASK_REGEX, str(p.relative_to(DATA_DIR)), re.I)]
_mask_set = set(mask_files)
image_files = [p for p in tifs if p not in _mask_set]
print(f"tif: {len(tifs)} | masks: {len(mask_files)} | images: {len(image_files)} | csv: {len(csvs)} | md: {len(mds)}")
print("example masks :", [str(p.relative_to(DATA_DIR)) for p in mask_files[:3]])
print("example images:", [str(p.relative_to(DATA_DIR)) for p in image_files[:3]])
if not mask_files:
    print("!! No masks recognised. Change MASK_REGEX in STEP 0 after looking at the file names above.")

summary = []
for p in mds[:6]:
    txt = p.read_text(errors="ignore")
    print("\n=== ", p.name, " ===\n", txt[:1500])
    summary.append(f"# {p.name}\n{txt[:1500]}")

import pandas as pd
meta = {}
for p in csvs[:3]:
    try:
        df = pd.read_csv(p)
    except Exception as e:
        print("CSV", p.name, "unreadable:", e); continue
    meta[p.name] = df
    print("\n=== CSV", p.name, df.shape, "===")
    print(list(df.columns)); print(df.head(3).to_string())
    summary.append(f"# CSV {p.name} {df.shape}\n{list(df.columns)}")
    for col in df.columns:
        if re.search(r"zone|area|site|region", col, re.I):
            print("counts by", col); print(df[col].value_counts().head(10).to_string())
        if re.search(r"date|time|acq", col, re.I):
            d = pd.to_datetime(df[col], errors="coerce")
            print("date range", col, d.min(), "->", d.max())
            print(d.dt.to_period("M").value_counts().sort_index().tail(12).to_string())

import rasterio
def describe(p):
    with rasterio.open(p) as ds:
        a = ds.read(1, out_shape=(min(ds.height, 256), min(ds.width, 256))).astype("float32")
        info = {"file": p.name, "crs": str(ds.crs), "bounds": tuple(round(v, 4) for v in ds.bounds),
                "bands": ds.count, "dtype": ds.dtypes[0], "size": (ds.height, ds.width),
                "min": float(np.nanmin(a)), "max": float(np.nanmax(a)), "mean": float(np.nanmean(a))}
    return info
for p in (image_files[:1] + mask_files[:1]):
    try:
        i = describe(p)
    except Exception as e:
        print("could not read", p.name, ":", e); continue
    print("\n", i); summary.append(str(i))
    if p in image_files[:1]:
        print("-> values look like", "dB" if i["min"] < 0 else "LINEAR power (will be converted to dB by to_db)")

img_keys = {norm_key(p): p for p in image_files}
pairs = [(img_keys[norm_key(m)], m) for m in mask_files if norm_key(m) in img_keys]
print("\nimage<->mask pairs found by name:", len(pairs), "of", len(mask_files), "masks")
print("dates found in file names:", sum(parse_date(p.name) is not None for p in tifs), "of", len(tifs))

Path("experiments").mkdir(exist_ok=True)
Path("experiments/inspect_summary.txt").write_text("\n\n".join(summary))

In [ ]:
# Event notes. Status column: what is confirmed and what you still must check yourself.
NOTES = '''# Event notes (Aya)

## Event
- Loukkos River / Ksar El Kebir and Gharb plain floods, Morocco, late January to February 2026.
- Radar scenes used by the team: Sentinel-1, before 21 Jan 2026, after 8 Feb 2026 (Path 147 / Frame 110).
- NISAR: no public data for these dates (searched by Ikram), so Sentinel-1 is used and we say so openly.

## Facts and status
| Fact | Value | Status |
|---|---|---|
| Flood onset | 28 Jan 2026 | Confirmed on GDACS (event 1103747) |
| GDACS alert level | Green (low humanitarian impact) | Confirmed on GDACS |
| Displaced (GDACS) | 20,000 (episodes 1-2), 50,000 (episode 3, 3 Feb) | Confirmed on GDACS |
| GDACS satellite window | 21 Jan to 10 Feb 2026 (observation window, not the end of the flood) | Confirmed on GDACS |
| Return of residents | 16 Feb 2026 | CHECK in Reuters article |
| Evacuated 188,000 / 143,164 (Interior Ministry) | | CHECK in AP / Reuters, delete if not found |
| Flooded area ~1,100 km2 (110,000 ha) | | CHECK source |
| Flooded area ~789 km2 at 8 Feb peak | | CHECK in the Springer study (a different measurement) |
| Dam release 372 million m3, 146% capacity, "Storm Leonardo", 3 billion MAD plan | | CHECK in AP / Reuters / Hespress, delete if not found |

## Sources
- GDACS: https://www.gdacs.org/Floods/report.aspx?detail=true&episodeid=1&eventid=1103747&eventtype=FL
- AP: https://apnews.com/article/f286b572a5f3029e99149b04c85b6c08
- Reuters: https://www.reuters.com/business/environment/morocco-residents-begin-returning-northwest-flood-waters-recede-2026-02-16/
- Hespress: https://en.hespress.com/130648-ksar-el-kebir-after-floods-damage-measures-and-whats-next.html

## Reference data
- SAR Morocco Flood Dataset, Zenodo, DOI 10.5281/zenodo.22964153, CC BY 4.0.
- 1,497 Sentinel-1 patches (256x256, VV+VH) with binary water masks; zone 1 Ksar El Kebir (102), zone 4 Gharb plain (887).
- Masks: HSBA + Otsu + morphology + visual QC (semi-automatic, not expert ground truth).
- Credit: Contains modified Copernicus Sentinel data 2024-2026, processed by Kzadri M., Begdouri A., El Fazazy K., Qjidaa H., Ouazzani Jamil M., Tairi H., Riffi J.
'''
Path("experiments/NOTES.md").write_text(NOTES)
print("written experiments/NOTES.md (open it and resolve every CHECK line)")

## STEP 5: Build the reference mask on Ikram's exact grid
Mosaics every mask patch that overlaps the scene (and is within the date window if a date is known) into `reference_mask.tif`:
**1 = water, 0 = not water, 255 = no reference here.**

In [ ]:
from rasterio.warp import reproject, Resampling, transform_bounds

with rasterio.open(BEFORE_TIF) as ds:
    G = dict(crs=ds.crs, transform=ds.transform, h=ds.height, w=ds.width, bounds=ds.bounds)
print("scene grid:", G["h"], "x", G["w"], G["crs"], tuple(round(v, 3) for v in G["bounds"]))

# optional dates from CSV metadata (id column + date column)
date_lookup = {}
for name, df in meta.items():
    try:
        stems = {norm_key(p) for p in tifs}
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and df[col].astype(str).map(norm_key).isin(stems).mean() > 0.5:
                for dcol in df.columns:
                    if re.search(r"date|time|acq", dcol, re.I):
                        d = pd.to_datetime(df[dcol], errors="coerce")
                        for k, v in zip(df[col].astype(str).map(norm_key), d):
                            if pd.notna(v): date_lookup[k] = v.date()
                break
    except Exception as e:
        print("csv date lookup skipped:", e)
print("dates known from CSV:", len(date_lookup))

after_d = dt.date.fromisoformat(AFTER_DATE)
gl, gb, gr, gt = G["bounds"]

def build_reference(use_date_filter):
    water = np.zeros((G["h"], G["w"]), np.uint8)
    covered = np.zeros((G["h"], G["w"]), np.uint8)
    used, skipped_date, no_date, no_overlap, unreadable = [], 0, 0, 0, 0
    for m in mask_files:
        try:
            with rasterio.open(m) as src:
                if src.crs is None:
                    unreadable += 1; continue
                l, b, r, t = transform_bounds(src.crs, G["crs"], *src.bounds)
                if r < gl or l > gr or t < gb or b > gt:
                    no_overlap += 1; continue
                d = parse_date(m.name) or date_lookup.get(norm_key(m))
                if d is None:
                    no_date += 1
                elif use_date_filter and abs((d - after_d).days) > WINDOW_DAYS:
                    skipped_date += 1; continue
                data = src.read(1)
                arr = (data > 0)
                if src.nodata is not None and src.nodata > 0:
                    arr &= (data != src.nodata)          # a positive nodata value is not water
                arr = arr.astype(np.uint8)
                ones = np.ones_like(arr, dtype=np.uint8)
                w_dst = np.zeros_like(water); c_dst = np.zeros_like(covered)
                reproject(arr, w_dst, src_transform=src.transform, src_crs=src.crs,
                          dst_transform=G["transform"], dst_crs=G["crs"], resampling=Resampling.nearest)
                reproject(ones, c_dst, src_transform=src.transform, src_crs=src.crs,
                          dst_transform=G["transform"], dst_crs=G["crs"], resampling=Resampling.nearest)
                water |= w_dst; covered |= c_dst
                used.append(m.name)
        except Exception as e:
            unreadable += 1
            print("skipped unreadable mask", m.name, ":", e)
    stats = dict(used=len(used), no_overlap=no_overlap, skipped_date=skipped_date, no_date=no_date, unreadable=unreadable)
    return water, covered, stats

date_filter_dropped = False
water, covered, st_ = build_reference(USE_DATE_FILTER)
if USE_DATE_FILTER and not covered.any():
    print(f"!! No patch overlaps the scene inside +-{WINDOW_DAYS} days of {AFTER_DATE}. Retrying WITHOUT the date filter: the reference may mix dates, say this in the report.")
    date_filter_dropped = True
    water, covered, st_ = build_reference(False)

print(f"patches used: {st_['used']} | outside scene: {st_['no_overlap']} | outside date window: {st_['skipped_date']} | "
      f"overlap but NO date known: {st_['no_date']} | unreadable: {st_['unreadable']}")
if st_["no_date"] and not date_lookup:
    print("!! Dates could not be read for some patches: the reference may mix dates. Say this in the report.")
assert covered.any(), ("No mask patch overlaps the scene, even without the date filter. Look at the bounds printed above "
                       "and in STEP 4: the dataset zones may not cover this scene. Say so in the report.")
ref = np.where(covered > 0, water, 255).astype(np.uint8)
print("reference covers", round(100 * (covered > 0).mean(), 2), "% of the scene; water share inside:",
      round(100 * water[covered > 0].mean(), 2), "%")

REF_TIF = Path("experiments/reference_mask.tif")
with rasterio.open(REF_TIF, "w", driver="GTiff", height=G["h"], width=G["w"], count=1, dtype="uint8",
                   crs=G["crs"], transform=G["transform"], nodata=255, compress="lzw") as dst:
    dst.write(ref, 1)
Path("experiments/reference_mask_info.json").write_text(json.dumps({
    "patches_used": st_["used"], "date_window_days": WINDOW_DAYS, "after_date": AFTER_DATE,
    "date_filter_dropped": date_filter_dropped,
    "patches_without_date": st_["no_date"], "covered_percent": float(100 * (covered > 0).mean()),
    "label_source": "Zenodo 10.5281/zenodo.22964153 (HSBA + Otsu, semi-automatic)"}, indent=2))
print("saved", REF_TIF)

In [ ]:
# Quick look at the reference on the scene
import matplotlib.pyplot as plt
with rasterio.open(AFTER_TIF) as ds:
    vv_after = to_db(ds.read(1))
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(vv_after, cmap="gray", vmin=-25, vmax=0); ax[0].set_title("AFTER VV (dB)")
show = np.ma.masked_where(ref == 255, ref)
ax[1].imshow(vv_after, cmap="gray", vmin=-25, vmax=0)
ax[1].imshow(show, cmap="winter", alpha=0.6, vmin=0, vmax=1); ax[1].set_title("Reference water (blue)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig("experiments/reference_overview.png", dpi=110); plt.show()

## STEP 6: Predictions and accuracy (IoU, F1) on the real event
* **Baseline:** Otsu threshold on the AFTER VV (computed here, no repo code needed).
* **CNN:** needs a water prediction raster from the repo. The cell below looks for one; if it does not exist it tells you the one Copilot task to run (task 1 in the Copilot prompt creates `experiments/predict_water.py`).

In [ ]:
from skimage.filters import threshold_otsu

with rasterio.open(AFTER_TIF) as ds:
    vv = to_db(ds.read(1)); vh = to_db(ds.read(2)) if ds.count > 1 else None
vv_after = vv                      # used by the failure crops below
valid_scene = np.isfinite(vv) & (vv > -60)
sample = vv[valid_scene][::50]
thr = float(threshold_otsu(sample))
base_water = ((vv < thr) & valid_scene).astype(np.uint8)
print(f"Otsu threshold on AFTER VV: {thr:.2f} dB | water pixels: {int(base_water.sum())}")

valid_eval = valid_scene & (ref != 255)
results = {"reference": "Zenodo SAR Morocco Flood Dataset (semi-automatic masks)",
           "after_date": AFTER_DATE, "otsu_threshold_db": thr}
results["baseline"] = scores(base_water, ref, valid_eval)
print("BASELINE:", {k: fmt(v) if isinstance(v, float) else v for k, v in results["baseline"].items()})

# CNN prediction raster: our own file first, then anything in the repo that looks like water-after
cnn_path = Path("outputs/aya_tmp/water_after_cnn.tif")
if not cnn_path.exists() and Path("experiments/predict_water.py").exists() and CKPT:
    rc = sh(f"PYTHONPATH=src:. {sys.executable} experiments/predict_water.py --tif {AFTER_TIF} --checkpoint {CKPT} --out {cnn_path}")
    if rc != 0:
        print("predict_water.py failed (see the message above). Fix it with Copilot, then re-run this cell.")
if not cnn_path.exists():
    guess = find_files(r"water_after.*\.tif$|cnn.*water.*\.tif$|pred.*after.*\.tif$")
    cnn_path = guess[0] if guess else None
print("CNN prediction raster:", cnn_path)

if cnn_path is not None:
    with rasterio.open(cnn_path) as ds:
        cnn_water = (ds.read(1) > 0).astype(np.uint8)
    assert cnn_water.shape == base_water.shape, f"CNN raster shape {cnn_water.shape} != scene {base_water.shape}: needs the same grid."
    results["cnn"] = scores(cnn_water, ref, valid_eval)
    print("CNN     :", {k: fmt(v) if isinstance(v, float) else v for k, v in results["cnn"].items()})
else:
    cnn_water = None
    print("\nCNN not scored yet. Do Copilot task 1 (predict_water.py), add it to the repo, restart this notebook so Colab clones it, then re-run from STEP 1.")

Path("experiments/event_metrics.json").write_text(json.dumps(clean_json(results), indent=2))

In [ ]:
# Team results already in the repo (from the model trainer): show them next to yours
for p in find_files(r"^eval_.*\.json$"):
    try:
        print(p.name, "->", json.dumps(json.load(open(p)))[:400])
    except Exception as e:
        print(p.name, "unreadable:", e)

## STEP 7: Failure cases (3 to 5 crops where the model is wrong)

In [ ]:
pred_for_cases = cnn_water if cnn_water is not None else base_water
who = "CNN" if cnn_water is not None else "Otsu baseline"
rows = tile_errors(pred_for_cases, ref, valid_eval)
picked = pick_tiles(rows)
print(f"{who}: saving {len(picked)} failure crops")
for f in Path("experiments/failures").glob("*.png"):
    f.unlink()
notes = [f"# Failure cases ({who}). Fill in what went wrong in each one.\n"]
for i, (kind, y, x) in enumerate(picked, 1):
    sl = (slice(y, y + 256), slice(x, x + 256))
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(vv_after[sl], cmap="gray", vmin=-25, vmax=0); ax[0].set_title("After VV (dB)")
    ax[1].imshow(vv_after[sl], cmap="gray", vmin=-25, vmax=0)
    ax[1].imshow(np.ma.masked_where(pred_for_cases[sl] == 0, pred_for_cases[sl]), cmap="autumn", alpha=0.7); ax[1].set_title(f"{who} water (red)")
    ax[2].imshow(vv_after[sl], cmap="gray", vmin=-25, vmax=0)
    ax[2].imshow(np.ma.masked_where(ref[sl] != 1, ref[sl]), cmap="winter", alpha=0.7); ax[2].set_title("Reference water (blue)")
    for a in ax: a.axis("off")
    name = f"experiments/failures/case{i}_{kind}_y{y}_x{x}.png"
    plt.suptitle(f"{kind.replace('_', ' ')} | tile y={y}, x={x}"); plt.tight_layout(); plt.savefig(name, dpi=100); plt.show()
    notes.append(f"## case{i} ({kind}) y={y}, x={x}\nWhat went wrong (sea? coast? tile seam? field? city? label error?): \n")
Path("experiments/failure_notes.md").write_text("\n".join(notes))
print("Open the PNGs in experiments/failures/ and write one sentence per case in experiments/failure_notes.md")

## STEP 8: Honest paragraph (a draft from real numbers only). Edit it in your own words.

In [ ]:
para = honest_text(results)
Path("experiments/HONEST_PARAGRAPH.md").write_text("# Honest paragraph (draft, edit it)\n\n" + para + "\n")
print(para)

## STEP 9: PDF report of your work
Builds `experiments/Aya_evaluation_report.pdf` from the real files above. Edit `EVENT_ROWS` if you verified more facts.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

EVENT_ROWS = [
    ["Flood onset", "28 Jan 2026", "Confirmed (GDACS)"],
    ["GDACS alert level", "Green, low humanitarian impact", "Confirmed (GDACS)"],
    ["Displaced (GDACS)", "20,000, later 50,000 (3 Feb)", "Confirmed (GDACS)"],
    ["Return of residents", "16 Feb 2026", "To check (Reuters)"],
    ["Evacuated, flooded area, dam release", "see notes", "To check (AP, Reuters, Springer study)"],
]
st = getSampleStyleSheet()
H1 = ParagraphStyle("h1", parent=st["Heading1"], textColor=colors.HexColor("#17375e"))
H2 = ParagraphStyle("h2", parent=st["Heading2"], textColor=colors.HexColor("#17375e"))
B = st["BodyText"]

def table(rows, widths):
    t = Table(rows, colWidths=widths, repeatRows=1)
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#dde7f3")),
                           ("GRID", (0, 0), (-1, -1), 0.4, colors.grey), ("FONTSIZE", (0, 0), (-1, -1), 9),
                           ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    return t

def cell(v): return Paragraph(str(v), B)
S = []
S += [Paragraph("Under the Clouds: Person 3 evaluation report", H1),
      Paragraph("Aya | Evaluation and Experiments | NASA Space Apps Challenge 2026, Meknes | Challenge: Dancing with the SARs", B), Spacer(1, 8)]
S += [Paragraph("1. Event and reference data", H2),
      table([["Fact", "Value", "Status"]] + [[cell(c) for c in r] for r in EVENT_ROWS], [5 * cm, 6.5 * cm, 5 * cm]),
      Spacer(1, 6),
      Paragraph("Reference: SAR Morocco Flood Dataset (Zenodo, DOI 10.5281/zenodo.22964153, CC BY 4.0), Sentinel-1 patches with water masks "
                "made by HSBA and Otsu thresholding. Contains modified Copernicus Sentinel data 2024-2026, processed by Kzadri M., Begdouri A., "
                "El Fazazy K., Qjidaa H., Ouazzani Jamil M., Tairi H., Riffi J.", B)]
info = json.load(open("experiments/reference_mask_info.json")) if Path("experiments/reference_mask_info.json").exists() else {}
S += [Paragraph(f"Reference mask on the team scene: {info.get('patches_used', 'n/a')} patches used, covering "
                f"{(info.get('covered_percent') or 0):.1f}% of the scene (date window +-{WINDOW_DAYS} days around {AFTER_DATE}; "
                f"{info.get('patches_without_date', 'n/a')} patches had no readable date)."
                + (" No patch fell inside the date window, so the date filter was dropped and the reference may mix dates." if info.get("date_filter_dropped") else ""), B)]
if Path("experiments/reference_overview.png").exists():
    S += [Image("experiments/reference_overview.png", width=16 * cm, height=6.6 * cm)]

S += [Paragraph("2. Accuracy on the Moroccan event (water after the flood)", H2)]
rows = [["Method", "IoU", "F1", "Precision", "Recall"]]
for key, label in (("baseline", "Otsu threshold baseline"), ("cnn", "U-Net (ResNet-34)")):
    r = results.get(key)
    rows.append([label] + ([fmt(r["iou"]), fmt(r["f1"]), fmt(r["precision"]), fmt(r["recall"])] if r else ["not scored yet"] + [""] * 3))
S += [table(rows, [5.5 * cm, 2.5 * cm, 2.5 * cm, 2.8 * cm, 2.8 * cm]), Spacer(1, 6),
      Paragraph("Earlier results from the Sen1Floods11 training data are in the repo (eval_valid.json, eval_test.json, eval_bolivia.json).", B)]

S += [Paragraph("3. Failure cases", H2)]
for img in sorted(Path("experiments/failures").glob("*.png"))[:5]:
    S += [Paragraph(img.stem.replace("_", " "), B), Image(str(img), width=15 * cm, height=5 * cm), Spacer(1, 4)]

S += [Paragraph("4. Honest paragraph", H2), Paragraph(Path("experiments/HONEST_PARAGRAPH.md").read_text().split("\n", 2)[-1], B)]
S += [Paragraph("5. Limitations", H2),
      Paragraph("Reference masks are semi-automatic (Otsu/HSBA) and may favour the threshold baseline. They are water masks from nearby dates, not a flood-change map. "
                "Flooded vegetation and shallow water look grey in radar and can be missed. Sentinel-1 is used because no public NISAR data exists for these dates.", B)]
S += [Paragraph("AI disclosure: an AI assistant (Claude) helped plan the evaluation and draft code and text; all numbers in this report were produced by running the code on the data above and must be checked by the team.", B)]

SimpleDocTemplate("experiments/Aya_evaluation_report.pdf", pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm,
                  topMargin=1.8 * cm, bottomMargin=1.8 * cm).build(S)
print("saved experiments/Aya_evaluation_report.pdf")
if IN_COLAB:
    from google.colab import files
    files.download("experiments/Aya_evaluation_report.pdf")

## STEP 10: Save your outputs (nothing is pushed from here)
Zips the `experiments/` folder and downloads it. Then, **on your computer in VS Code**: unzip it into the repo's `experiments/` folder, check the files, and commit and push them yourself.
Colab sessions are wiped when they close, so a copy is also saved to your Google Drive (`aya_experiments_backup`). Never commit `data/` or `*.pt`.

In [ ]:
zip_path = shutil.make_archive("aya_experiments", "zip", "experiments")
print("created", zip_path, round(Path(zip_path).stat().st_size / 1e6, 2), "MB")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copytree("experiments", "/content/drive/MyDrive/aya_experiments_backup", dirs_exist_ok=True)
        print("copied to Drive: aya_experiments_backup")
    except Exception as e:
        print("Drive backup skipped:", e)